# E2 · Batería de artefactos

**Spec:** [`docs/spec_E2_codex_artifact_tests.md`](../docs/spec_E2_codex_artifact_tests.md)  |  **Bloque:** E · Resultado  |  **Run de este set:** `ROXs12b_realigned`

Batería de tests de artefactos (T1–T5) para caracterizar la no-detección.

| | |
|---|---|
| **Entrada** | Productos de extracción |
| **Salida (QC/productos)** | `stages/stage_h02_qc.json` |
| **Consume aguas abajo** | E3, F1 |


## Qué hace E2 y cómo se reinterpreta

E2 corre una **batería FIJA de 5 tests de artefactos** sobre el resultado de E1 (siempre completa, gane o no E1). Para una no-detección, la pregunta es: **¿es robusta?**

- **T1 (coincidencia** con stripe/skyline/laser): `unavailable` (no hay listas — exposición única, sin QC de stripes).
- **T2 (forma de PSF del máximo global):** el máximo de Hα **NO** se ajusta mejor con una PSF que con un plano (`chi2_ratio` 1.007 ≈ 1) y está **elongado** (1.54, no puntual). status `fail` — pero para una **no-detección** esto se **reinterpreta** (spec §2): que el máximo NO tenga forma de PSF significa que es **ruido**, no una fuente → **apoya la no-detección**.
- **T3 (split de exposición):** `unavailable` (exposición única).
- **T4 (variación de knobs):** `unavailable` (variantes de validación no generadas).
- **T5 (placebos):** buscar en λ **fuera de línea** (6200/6400/6700/7100 Å) → **sin detección espuria** (max_fap 0.029, ninguno <0.01). **PASS**: el método no fabrica detecciones en λ aleatorios.

**`overall_raw = fails`** (driven por T2) → **reinterpretado a `overall = survives`** (`non_detection_robust`). Es una **limitación aceptada** documentada en F1 (la reinterpretación de T2 para no-detección).


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
bash scripts/stage_h02_artifacts.sh --run-id $RUN
```

Ligero.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage_h02_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'bash scripts/stage_h02_artifacts.sh --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage_h02_qc.json', RUN_ID)
nb.show(qc, keys=['overall', 'overall_raw', 't2.status', 't5.status', 'halpha_map_correlation.corr_stripe_scatter_vs_halpha_sigma', 'halpha_map_correlation.halpha_sigma_slicer_aligned'], title='E2')


## La batería T1–T5, en físico

`t1`…`t5` son la batería **congelada** de la spec: cinco maneras distintas de que lo que E1 midió sea un artefacto y no el compañero. Cada una ataca un origen instrumental diferente, y por eso se pasan todas aunque E1 diga no-detección (en ese caso se aplican al máximo del mapa, o sea al ruido dominante).

| Test | ¿Qué artefacto descarta? | Criterio |
|---|---|---|
| **T1 · coincidencia instrumental** | Que el canal de la señal caiga sobre algo que el instrumento ya ensucia: franjas del slicer (lista de B2), líneas de cielo (catálogo de A4) o los bordes del hueco del láser AO. | Distancia al artefacto más cercano de cada lista; coincide si \|Δcanal\| ≤ 2. |
| **T2 · coherencia espacial** | Que la señal no tenga la forma de una estrella. Una fuente real es la PSF de C1 centrada donde B3 puso al compañero; un artefacto es alargado, desplazado o con varios picos. | χ² de PSF vs plano, centroide < 1 px de B3, elongación < 1.5×. |
| **T3 · estabilidad temporal** | Que venga de una sola exposición. Una señal real crece como √N al combinar; un rayo cósmico o un defecto no. | z > z_total/√2 en ambas mitades independientes. **Con una sola exposición no se puede aplicar** — queda `unavailable` y así consta: es un eje de robustez que este dataset no cubre. |
| **T4 · estabilidad frente a parámetros** | Que la señal dependa de cómo la analizamos. Se repite E1 moviendo UNA perilla cada vez (radio de ajuste local, máscara de C1, ancho de plantilla, continuo). | Rango de z entre variantes Δz < 1. |
| **T5 · placebos espectrales** | Que el método «detecte» en cualquier sitio. La cadena completa se centra en líneas donde no se espera nada (6200, 6400, 6700, 7100 Å; 6300 descartada por skyline). | Ningún placebo supera el umbral de E1. Si alguno lo supera, **la FAP está mal calibrada** y el resultado de E1 no vale. |


## Resultados que llevaron a la conclusión

Los 5 tests con su status y la reinterpretación del `stage_h02_qc.json`.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('E2', 'stages/stage_h02_qc.json'):
        q = nb.load_qc('stages/stage_h02_qc.json', RUN_ID)
        print('E1 input:', q['input_verdict_e1'], '| método seleccionado:', q['selected_method'])
        print(f"T1 coincidencia: {q['t1']['status']}")
        t2 = q['t2']
        print(f"T2 forma-PSF del máximo: {t2['status']}  (chi2_ratio_psf_vs_plane={t2['chi2_ratio_psf_vs_plane']:.3f}, "
              f"elongación={t2['elongation_vs_psf']:.2f}) -> NO es PSF -> apoya no-detección")
        print(f"T3 split exposición: {q['t3']['status']} ({q['t3'].get('reason','')})")
        print(f"T4 variación knobs: {q['t4']['status']} ({q['t4'].get('reason','')})")
        print(f"T5 placebos: {q['t5']['status']}  (max_fap={q['t5']['placebo_max_fap_global']:.3f}, any_above={q['t5']['any_above_threshold']})")
        print()
        print(f"overall_raw = {q['overall_raw']}  ->  overall = {q['overall']}")
        if q['overall_raw'] != q['overall']:
            print(f"interpretación: el veredicto crudo ({q['overall_raw']}) se revisa a "
                  f"{q['overall']} por los tests que sí aplican; ver t1..t5 arriba.")
        else:
            print(f"interpretación: veredicto consistente ({q['overall']}).")


## Plot 1 — T5 placebos: el método no inventa detecciones

El `z` del matched filter buscando en centros **fuera de línea** (6200–7100 Å) por método, y el **Hα real** (★) en 6562.8. El Hα real cae en el **mismo nivel de ruido** que los placebos → no hay detección espuria y la no-detección es robusta.


In [ ]:
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    rd = nb.run_dir(RUN_ID)
    q = nb.load_qc('stages/stage_h02_qc.json', RUN_ID)
    df = pd.DataFrame(q['t5']['rows'])
    centers = sorted(df['center_A'].unique())
    methods = ['aperture', 'optimal_psfsub', 'psffit', 'optimal_ls']
    real = pd.read_csv(rd / 'tables' / 'halpha_detection_by_method.csv').set_index('method')['matched_z']
    cmap = dict(zip(methods, ['tab:blue', 'tab:green', 'tab:red', 'tab:orange']))
    fig, ax = plt.subplots(figsize=(9, 4.3))
    for m in methods:
        zc = [df[(df.center_A == c) & (df.method == m)]['matched_z'].values[0] for c in centers]
        ax.plot(centers, zc, 'o-', color=cmap[m], ms=6, label=f'placebo {m}')
        ax.scatter([6562.8], [real[m]], marker='*', s=160, color=cmap[m], edgecolor='k', zorder=5)
    ax.axvline(6562.8, color='0.5', ls=':', label='Hα real (★)')
    ax.set_xlabel('centro de búsqueda [Å]'); ax.set_ylabel('z del matched filter')
    ax.set_title('E2 · T5 placebos: λ off-line da ruido; Hα real (★) igual → sin detección espuria')
    ax.legend(fontsize=7, ncol=2); fig.tight_layout()
    outdir = rd / 'plots' / 'e2_artifacts'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 't5_placebos.png', dpi=110); print('figura ->', outdir / 't5_placebos.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — la batería a golpe de vista

Status de los 5 tests. T2 en rojo (`fail`, pero reinterpretado: el máximo no es PSF → apoya la no-detección); T5 en verde (placebos limpios); T1/T3/T4 en gris (no disponibles por la exposición única).


In [ ]:
try:
    import matplotlib.pyplot as plt
    q = nb.load_qc('stages/stage_h02_qc.json', RUN_ID)
    tests = {'T1 coincidencia\n(stripe/skyline/laser)': q['t1']['status'],
             'T2 forma PSF\ndel máximo': q['t2']['status'],
             'T3 split de\nexposición': q['t3']['status'],
             'T4 variación de\nknobs': q['t4']['status'],
             'T5 placebos\n(λ off-line)': q['t5']['status']}
    col = {'pass': 'tab:green', 'fail': 'tab:red', 'unavailable': '0.7'}
    names = list(tests)
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.barh(names, [1] * len(names), color=[col.get(tests[n], '0.7') for n in names])
    for i, n in enumerate(names):
        ax.text(0.5, i, tests[n], ha='center', va='center', fontsize=9, color='w', weight='bold')
    ax.set_xlim(0, 1); ax.set_xticks([]); ax.invert_yaxis()
    ax.set_title(f"E2 · batería: overall_raw={q['overall_raw']} -> {q['overall']} (T2 reinterpretado)")
    fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'e2_artifacts'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'battery.png', dpi=110); print('figura ->', outdir / 'battery.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 3 — S1b: zonas sucias de stripes vs mapas Hα (Xie+20 §4.1)

Correlación por columna entre la perturbación de la solución de onda (scatter del mapa de offset S0) y el ancho σ del Hα (mapa S1). La clave es el **control transversal**: si la correlación en la dirección transversal es casi igual, la correlación es un **confundido radial** (núcleo brillante vs halo débil), NO una firma de slicer. `halpha_*_slicer_aligned` aplica el mismo criterio que el gate G1 (estructura > 3× y > 2× su control transversal).


In [ ]:
try:
    from IPython.display import Image, display
    q = nb.load_qc('stages/stage_h02_qc.json', RUN_ID)
    hmc = q.get('halpha_map_correlation')
    if not hmc:
        print('S1b no integrado en este run: falta halpha_map_correlation.')
        print('-> corre: python scripts/s1b_integrate_e2.py --run-dir', nb.run_dir(RUN_ID))
    else:
        print(f"corr(stripe scatter, Hα σ)      = {hmc['corr_stripe_scatter_vs_halpha_sigma']:.3f}")
        print(f"  control transversal            = {hmc['corr_stripe_scatter_vs_halpha_sigma_transverse']:.3f}  (≈ igual ⇒ confundido radial)")
        print(f"a  estructura {hmc['halpha_a_structure_significance']:.1f}× (transv {hmc['halpha_a_transverse_significance']:.1f}×) -> slicer_aligned={hmc['halpha_a_slicer_aligned']}")
        print(f"σ  estructura {hmc['halpha_sigma_structure_significance']:.1f}× (transv {hmc['halpha_sigma_transverse_significance']:.1f}×) -> slicer_aligned={hmc['halpha_sigma_slicer_aligned']}")
        print(f"corr(a,σ) = {hmc['halpha_corr_a_sigma']:.2f}, P_cov = {hmc['halpha_P_cov']:.2f}  (Xie Fig.3 pide P≈const)")
        fig = (q.get('figures') or {}).get('s1_stripe_halpha')
        from pathlib import Path as _P
        if fig and _P(fig).exists():
            display(Image(filename=str(fig)))
except Exception as e:
    print('No se pudo mostrar S1b:', type(e).__name__, e)


## Decisiones y notas
- **S1b (wavesol/stripes):** los mapas Hα a/σ NO están alineados con slicers (estructura ≤ control transversal ⇒ radial, núcleo vs halo); la fuerte correlación por columna stripe↔σ es un confundido radial (transversal ≈ igual). Consistente con G1 (cubo combinado ciego a stripes). Sin interpretar ghost-vs-instrumental (humano/por-exposición). · [`docs/2026-07-17_decision_g1_wavesol.md`](../docs/2026-07-17_decision_g1_wavesol.md)
- **T2 reinterpretado para no-detección** (spec §2): el máximo global NO tiene forma de PSF (chi2_ratio 1.007, elongación 1.54) → *apoya* la no-detección. `overall_raw=fails` → `overall=survives`. Limitación aceptada en F1.
- **T5 placebos PASS**: buscar en λ off-line no fabrica detecciones (max_fap 0.029, ninguno <0.01) → método limpio.
- T1/T3/T4 `unavailable` por la exposición única (sin stripe QC, sin split, sin variantes de knobs).


## Conclusión (registrada)

**E2: la no-detección es ROBUSTA (`overall=survives`, reinterpretado de `fails`).**

- **Fecha:** 2026-07-09 (re-run con PSF Psfao).
- **T2** `fail` reinterpretado: el máximo de Hα no es PSF (chi2_ratio 1.007, elong 1.54) → ruido → apoya la no-detección.
- **T5** PASS: placebos en λ off-line sin detección espuria (max_fap 0.029).
- **T1/T3/T4** `unavailable` por la exposición única.
- **F1:** la reinterpretación de T2 es una **limitación aceptada** documentada (no un rojo bloqueante).
- **Downstream:** con la no-detección robusta, E3 calcula el límite superior de Ṁ.
